<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 1: Instances, Internals, and Plugins</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">The plumbing under the porcelain, and how to extend it</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

Now that we have covered the basic commands and hierarchical scheduling, let's look at
the structure of an individual Flux instance and the services that make it run:

1. Process and monitoring utilities
2. The structure of Flux instances
3. `flux kvs`, which powers a lot of the higher level commands
4. `flux archive` for sites without a shared filesystem
5. Writing your own job validator plugin, in Python

```bash
cd /home/ubuntu/tutorial/module1
```


# Process, Monitoring, and Job Utilities ⚙️
## flux exec 👊️

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Executing commands across ranks
</div>

Have you ever wanted a quick way to execute a command to all of your nodes in a flux instance? It might be to create a directory, or otherwise interact with a file. This can be hugely useful in environments where you don't have a shared filesystem, for example. This is a job for flux exec! Here is a toy example to execute the command to every rank (`-r all`) to print.

```bash
flux exec -r all echo "Hello from a flux rank!"
```

You can also use `-x` to exclude ranks. For example, we often do custom actions on the main or "leader" rank, and just want to issue commands to the workers.

```bash
flux exec -r all -x 0 echo "Hello from everyone except the lead (0) rank!"
```

Here is a similar example, but asking to execute only on rank 2, and to have it print the rank.

```bash
flux exec -r 2 flux getattr rank 
```

And of course, we could do the same to print for all ranks! This is a derivative of the first example we showed you.

```bash
flux exec flux getattr rank
```

You can imagine that `flux exec` is hugely useful in the context of batch jobs and machine learning leader/worker designs that need different commands to start or interact with component nodes.

## flux jobs

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Querying all jobs on a cluster
</div>

Flux provides a way to query information about all of the jobs on a cluster, which was discussed in Chapter 1. We sort of skipped over the powerful ways through which you can customize this command. As a reminder, the basic usage to see current and "all" jobs:

```bash
# Show me currently active jobs
flux jobs

# Show me ALL jobs
flux jobs -a
```

You can remove the header and ask for a format string to get exact information. Here is how to get a list of status:

```bash
flux jobs -a --no-header --format="{status}"

# Get one status
flux jobs --no-header --format="{status}" $(flux job last)
```


## flux job info

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Querying information about a Flux job
</div>

Flux provides a way to get information from a JOBID, like jobspec, eventlog, and R (resource set).

```bash
echo "Querying the jobspec"
flux job info $(flux job last) jobspec | jq .resources
echo "Querying the resource set"
flux job info $(flux job last) R | jq .execution.nodelist
Querying the jobspec
[
  {
    "type": "node",
    "count": 1,
    "with": [
      {
        "type": "slot",
        "count": 64,
        "with": [
          {
            "type": "core",
            "count": 1
          }
        ],
        "label": "task"
      }
    ]
  }
]
Querying the resource set
[
  "ip-10-0-25-10"
]
```

We can query the job event log:

```bash
flux job eventlog -H $(flux job last)
[Aug09 01:10] submit userid=1000 urgency=16 flags=0 version=1
[  +0.011880] validate
[  +0.022558] depend
[  +0.022598] priority priority=16
[  +0.024352] alloc
[  +0.026081] start
[  +0.042162] memo uri="ssh://ip-10-0-25-10/tmp/flux-JE2tV6/local-0"
[  +1.491008] finish status=0
[  +1.491826] release ranks="all" final=true
[  +1.491856] free
[  +1.491869] clean
```
Or the exec event log:

```bash
[Aug09 01:10] init
[  +0.009753] shell.init service="1000-shell-f31zWKSBmy" leader-rank=0 size=1
[  +0.011289] shell.start taskmap={"version":1,"map":[[0,1,1,1]]}
[  +0.000687] starting
[  +1.464679] shell.task-exit localid=0 rank=0 state="Exited" pid=72151 wait_status=0 signaled=0 exitcode=0
[  +1.466305] complete status=0
[  +1.466328] done
```

## flux uptime

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Showing how long a flux instance has been running
</div>

Flux provides an `uptime` utility to display properties of the Flux instance such as state of the current instance, how long it has been running, its size and if scheduling is disabled or stopped. The output shows how long the instance has been up, the instance owner, the instance depth (depth in the Flux hierarchy), and the size of the instance (number of brokers).

```bash
flux uptime
 01:52:57 run 5h,  owner ubuntu,  depth 0,  size 1
```

## flux top 

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Showing a table of real-time Flux processes
</div>

Flux provides a feature-full version of `top` for nested Flux instances and jobs. Try it out! 

```bash
flux top
```

If you don't have any jobs running, run a few `flux submit sleep inf` to see output. 

## flux proxy

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Interacting with a job hierarchy
</div>

Flux proxy is used to route messages to and from a Flux instance. We can use `flux proxy` to connect to a running Flux instance and then submit more nested jobs inside it. This can work with flux uris that start with `ssh://` or even local files! Let's do a dummy example to reconnect to our own uri:

```bash
flux proxy $FLUX_URI flux resource list
```

In the case that was another (remote) Flux instance or a batch job, we can easily execute commands.

## flux queue

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Interacting with and inspecting Flux queues
</div>

Flux has a command for controlling the queue within the `job-manager`: `flux queue`.  This includes disabling job submission, re-enabling it, waiting for the queue to become idle or empty, and checking the queue status:

```bash
flux queue enable
flux queue -h
```

## flux getattr

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Getting attributes about your system and environment
</div>

Each Flux instance has a set of attributes that are set at startup that affect the operation of Flux, such as `rank`, `size`, and `local-uri` (the Unix socket usable for communicating with Flux).  Many of these attributes can be modified at runtime, such as `log-stderr-level` (1 logs only critical messages to stderr while 7 logs everything, including debug messages). Here is an example set that you might be interested in looking at:

```bash
# What is the rank of the node I'm sitting on?
flux getattr rank

# What is the size of the Flux instance?
flux getattr size

# What is the local uri?
flux getattr local-uri

# Set an attribute for the log level
flux setattr log-stderr-level 3

# List current attributes
flux lsattr -v
```

## flux dmesg

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Viewing Flux system messages
</div>


If you need some additional help debugging your Flux setup, you might be interested in `flux dmesg`, which is akin to the [Linux dmesg](https://man7.org/linux/man-pages/man1/dmesg.1.html) but delivers messages for Flux.

```bash
flux dmesg -H
```

<br>

## Flux Module

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Manager and query Flux modules.
</div>


To manage and query modules, Flux provides the `flux module` command. This is more of an advanced use case. The sub-commands provided by `flux module` can be seen by running:

```bash
flux module --help
```

We can see that these services are loaded and available by running:

```bash
flux module list
```

Some examples of Flux modules include:
* `job-ingest` (used by Flux submission commands like `flux batch` and `flux run`)
* `job-list` (used by `flux jobs`)
* `sched-fluxion-qmanager` (used by `flux tree`)
* `sched-fluxion-resource` (also used by `flux tree`)

Try looking at job manager stats, and then using `jq` to get a specific value.

```bash
flux module stats job-manager
flux module stats job-manager | jq .inactive_jobs
```

Users and system administrators can easily load and unload modules using the `flux module load` and `flux module remove` commands. 

### flux kvs

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> The Flux key value store.
</div>

One of the core services built into Flux is the key-value store (KVS). It is used in many other services, including most of Flux's resource management services, and the `flux archive` service below.

The `flux kvs` command provides a utility to list and manipulate values of the KVS. As a example of using `flux kvs`, let's use the command to examine information saved by the `resource` service.

```bash
flux kvs ls
flux kvs ls resource
flux kvs get resource.R | jq
```

The KVS is such an essential component of Flux that we provide C and Python APIs to interact with it. To learn more about interacting with the KVS from these languages, take a look at these documentation pages:
* C's `flux_kvs_commit` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_commit.html)
* C's `flux_kvs_copy` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_copy.html)
* C's `flux_kvs_getroot` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_getroot.html)
* C's `flux_kvs_lookup` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_lookup.html)
* C's `flux_kvs_namespace_create` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_namespace_create.html)
* C's `flux_kvs_txn_create` [family of functions](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/man3/flux_kvs_txn_create.html)
* Python's `flux.kvs` [module](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/python/autogenerated/flux.kvs.html#module-flux.kvs)

## flux archive 📚️

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Description:</span> Creating file and content archives to access later and between ranks
</div>

As Flux is used more in cloud environments, we might find ourselves in a situation where we have a cluster without a shared filesystem. The `flux archive` command helps with this situation. At a high level, `flux archive` allows us to save named pieces of data (e.g., files) to the Flux KVS for later retrieval.

When using `flux archive`, we first have to create an named archive. In the code below, we will create a text file and then save it into an archive using `flux archive`. Note that, for larger files, you can speed up the creation and extraction of archives by using the `--mmap` flag.

<br>

# 🔧 Plugin Workshop: Write a Job Validator

<div class="alert alert-block" style="background-color:#D9A441;color:#06293D">
<span style="font-weight:600">Description:</span> Extending Flux in Python. No compiler, no root, no restarting the instance.
</div>

Flux has several extension points, and they are not all the same language:

| Extension point | Language | What it controls |
|---|---|---|
| **Job validator** | **Python** | Whether a job is admitted at all |
| Job shell plugin | Lua or C | What happens when tasks launch |
| Jobtap plugin | C only | Scheduling policy and priority |
| `flux` subcommand | Anything executable | New commands |

We are doing the Python one. Every job submitted to this instance passes through
the validator before it reaches the scheduler, so this is where site policy lives:
core limits, banned commands, "please use flux batch for anything this big."

A validator plugin is one class with one required method:

```python
from flux.job.validator import ValidatorPlugin

class Validator(ValidatorPlugin):
    def validate(self, job):
        # return None to accept, or (errno, message) to reject
```

The `job` argument gives you `job.jobspec` (the submitted jobspec as a plain dict),
plus `job.userid`, `job.flags`, and `job.urgency`. To count resources, hand the
jobspec to `JobspecV1` and call `resource_counts()`, which returns a dict like
`{"node": 1, "slot": 4, "core": 8}` with nested counts already multiplied out.

## The skeleton

Open [plugin-workshop/pizza_policy.py](plugin-workshop/pizza_policy.py). The class and
the resource lookups are written; the decision is yours.

## Test it without touching the instance

`flux run --dry-run` prints a jobspec instead of submitting it, so you can pipe one
straight into the validator. This is a fast edit-and-rerun loop:

```bash
cd /home/ubuntu/tutorial/module1
flux run --dry-run -n4 sleep 60 | flux job-validator --jobspec-only --plugins=./plugin-workshop/pizza_policy.py
```

A passing job prints an errnum of 0. A rejected one prints your message.

Try the worked example, which caps cores and refuses a couple of toppings:

```bash
flux run --dry-run -n8 sleep 60 | flux job-validator --jobspec-only --plugins=./plugin-workshop/oven_capacity.py --max-cores=4
```

```console
{"errnum": 22, "errstr": "order needs 8 cores, the oven fits 4"}
```

## Then load it for real

```bash
flux module reload job-ingest validator-plugins=jobspec,$(pwd)/plugin-workshop/oven_capacity.py
```

Now ordinary submission goes through it:

```bash
flux submit -n8 sleep 60
```

Put things back when you are done:

```bash
flux module reload job-ingest validator-plugins=jobspec
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Your turn:</span> Cap the core count. Reject a command by name and say why. Require every job to have a <code>--job-name</code>. Only accept work during business hours. Reject anything asking for zero nodes, on principle. We will share the best ones at the start of Module 3.
</div>

<!-- TODO(sc26): smoke-test against the tutorial image. The interface is from
     flux-core v0.86.0 (the version build-ubuntu.sh installs): ValidatorPlugin and
     ValidatorJobInfo in src/bindings/python/flux/job/validator/validator.py, and the
     validator-plugins=<path> form is exercised by t/t2110-job-ingest-validator.t.
     Both plugin files were tested against the basic_v1.yaml jobspec fixture with a
     faithful port of resource_counts, but not yet against a live job-ingest module.
     The notebook runs under `flux start --test-size=4`, so confirm whether the module
     reload needs `flux exec -r all -x 0` as the flux-core test helper does. -->

## If you would rather write Lua

The job shell is the other user-facing extension point, and it takes inline Lua plugins
registered with `plugin.register` in an initrc. Where the validator decides *whether* a
job runs, a shell plugin decides *what happens* when its tasks launch.

[plugin-workshop/oven-timer.lua](plugin-workshop/oven-timer.lua) is a skeleton and
[plugin-workshop/order-ticket.lua](plugin-workshop/order-ticket.lua) is a complete
example. Neither needs installing:

```bash
flux run -o verbose=2 -o userrc=plugin-workshop/order-ticket.lua -n2 hostname
```

## Where to read more

- `flux-job-validator(1)` and `flux-config-ingest(5)` for the validator
- `flux-shell-initrc(5)` for the Lua shell plugin API
- `flux-jobtap(1)` for scheduling policy, if you are comfortable in C
- `flux-environment(7)`, `FLUX_EXEC_PATH` &mdash; drop an executable named `flux-something`
  on that path and it becomes `flux something`, in any language you like


<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 1 complete</strong></div>

Continue to [Module 2: Converged Environments](../module2/01_flux_operator_cloud.ipynb).
